# 修正地图上的姿态图乘法 CUDA 配对验证

选用私有 Notebook 和 GPU，挂载修正 BEV 坐标的共用私有输入 `pinns_sync_latency_source_label_private.bin`，Internet 可关闭。全 72 格逐值核对已训练模型的六个候选、动作输出、排序和近并列，配对计时相同输入上 A/B 两种执行。主要时延比较预定为 20 帧、12 人的两个真实时刻，对每场的配对节省量中位数取等权平均。正负结果均保留。

输出 `/kaggle/working/pinns_paired_pose_compute_source_label.zip` 包含可复查的 `paired_report.json` 和完整的私有 `private_outputs.npz`。后者包含逐行模型结果，请仅在私有环境核对。

In [ ]:
from pathlib import Path
import base64, hashlib, subprocess, sys, tempfile, zlib
code_dir = Path(tempfile.mkdtemp(prefix="pose-graph-code-"))
sources = {'paired_pose_compute.py': ['83c07be96784fc237227b2f29c330b002b81adda7a65283e26ea216f4c2dd478', 'eNqVO2tT40iS3/kVtYq4C2nWCNtgmubWE0fT9Ezf9gBB07sb53MoSlLZqJEljyQDhuN+++WjSm/3YyKmseuRlZXvzEpblnUto0yF4vzL+zMhN2FUiHQhijslZBbcRQ8wtdjEsVjDoigo0kzIJBRRkYuvaZQUB8GdTBIVi0zJPE+DSBZRmriWZe1Fq3WaFQBmuZZZrvYWWboSoSxUEa2U0LPm+0Dgv89posy+O5nfxZFvvvIfGHA3RRSXo2n5qVBZkaZxbga+5mliPqc5n76WBcI0h1/DV7Mkg2ulK/OtUKv1IopLZBA58/k54inzPdms1lshc5Gsy+Up0G5vb+/dl8v3ny7EVFjrKElyL98mgRfDlZNg6+XpJgsUfPVV7K2z6AHGXT9KLL3N+/z7GW6V/pvD4Yn0h6PjcBjIcDjxfRUE4+OTcLiQ4UiNRoEcBb4aB4vh8bGvjtVwFCo5fHN0PFGBb+3dfLm8vLgx8E6Ohm/l0ZvJ4mQUHr09ksPxoZTHweHRKDjxj8Ij/01wOJoMg8lwGPgnUobK9+Xo6CQ4UfJkeHJk7d3enF1+/nB184cBeTicvJWT4RjQHL7x3xyP3x4t3hwfLhDEaHh0uAjU8SEAfzscH0/eHJ4M1Vt5LOVkdHLiL9T40Nq7vrl6591efQJoY7U/wSP+q/p+tHd5cXbj/XZ2jd/FL8Is37u5uL44u/0Mw6PhcO+fZzd/fLnGb+PJ3ueLi/f4aTiejEbw/eJf1xfntzT2YhWZ/Jpbp8I+HojD4UCMnYGwVmmogBOpr2fKsThdgsSbwT1R+8+SQVHtmdAeHKq28NgSuOs9qmh5x6OHNJr4mSeLIsGR0XjgvILQhGoB2vTnBtTSTu8HYpUvnVM6M1qIJAVpvj8tUchklCvxDxlv1EWWpZmNqzWQ/E7aoF9S785UsckSo1cuzI4nx7zAvVNPYbRUeWGXmwu5VPYq3STFQCzSOFQZ6ChoSb5Is5UGma5R23OgaA5ir0Je72bLOPVtlmLH0YfrC+kdaERkHNuI49oF4xF6/rZQue04YjoVNQ2A48RaRIk5rEP/opDBHVmsIM0yFQAegnVrn3RLaN0SqFz/gaeKlSzAtiVLgYqci9UmR3vzoMiuZWoZ5WBNAAycbzXxR3xLKnTxbmgG8HeTqKc1o7ROgVFRsgDASaA0hhr6Y1TcGbvi/ne0/gB/Dalmw7mD1uW5xnSNzLMLJxewz3ZElItLsJ5wZpSsN4U4vzkXCxnFm8wcgv89u+oJ0A8KpD1zlSfBPD6rBBjJg+IAJB8sOuw29spdb60OKXhblw6VxWkSIc2iZZRIcBcb8BqZxiyHWTi6aeJdHPXQbHtIFlCogHyLbfGhDWNqDfQNGGB5TAcoqPMGgBFYPMDGfyos3DgFe5cBleBsXmsbYLzKrhEITEN8kGfBAX3KD5DFXsliJJjjPmbgmTRtdghOXTvNYVoLgzsV3Hugi7jSzA1IFVbRswq1HiZypVALX16ZmaAyODQQciB81J2ZbRl3ztCAYAacSwMeenfbqcFujLe0rvYfQI5jz98s4No5gKX1JWxEIzSzCEbPl4e0FzjzrpxL915tYRIly9efgeHWC25+1bfHQGSpwpqwIxnu8fbytIF8CXZ2P0czuFYEuPpGxgm/hsW2NsnfcJL8uwtgZExQBrSghtOpeLl/7UGJUJ3hv3NgV6wSWzbtCy9oM2uOOIyOR01lIuFYYxQmaJnIgaJWQ5wInLHom9VKZrDEA5UBe20/DcQWbHoaK5DLQFVO5knT4S9TseWPdZ4Q4BfLsNijBeDAYrCatt6Kvq3kcXPFtlqBZi9KvBIDWPJBxrliKSbZnQIyMkfC28naXYB6FsdHsHfbM8qyHyWgcLAPJqKcvwGRxb83BnxeHPI66efARp6ai31gpv7coOWL9YBuFr23dJHU6L/ThJd6awijcQq4Ydv/x4OOCzTv6I4VRgsyEYWn95ZwaXOIdB+WezEATB9Ug0p63a815vUftZJPnqTQhIhkhy6MoI0GLod0CQGGSxnXUT+j/NwC2cM0H+Jum+/iomNxSEkAwb9N6xjSVBXiyOQ+JxnUgicpkCAqR6GXpY+I9pAcWmeoSNcjz6gXD2GW4qUZ2ObGRB35SmbVE8ZoAMUrImWAVgLbP202J0pmzcnmISWY7sIm5p6mJa1bynUXVt9qCDVVhrZh/OQhf3m6opXCSKyUx+5g36mV3zAeA6OKH1c9pzIQOki1m/pXk4umHmqpaFpoEIVZk+1z8VewgI1FQZoUUbJRtVh4IDJfq3S2xJDU3oex+ygJpxaYSD8G8zhoTPvt6RJaCDtDhCZnmYQgDA0DfRzNyd5nvh6kjzC418B+N+40vUsSaS2qdyjR5g+d1rZdIlrt83v39ctu/TTQVZNk7Tx0127/G7t79LLciIJC3Mjk1mN3ykx0nLo8Ef3RKhLNW6ISFLAN5bjDKmJQV64atqNHrjqLOgpT4l87vJcA3wS3Q493AMesF0iF5tt4rH0gU3XWn0C4P1vSn8mmuGd+MzqKBuIrKntZPHGDdAVZEgXaOWwHhGyOUhzMkZukB87Yf8pZNBd/g9NnXyFPAR7Zf/p6yMeh007caJS7bqd62LBj6TeYQV48YuYjMh2WaEcOUE14na7WEiNBUGgglEZWYSKdw9AWHLSKMUiYgcq3gmukm856c1VgiPG/Aj/45oOpN7SNI+5FwcfAVBhgesBvDxggTTIyhq5cr1US2lXQuYryHNNaAFJPujYFZISW823riZ6YVIiiU7Rw9KGOuky2kHNyGPwXEjSy/4djPPC5ChrtZCB+MZgzGIfo9oxXYp/v/NiN6LADhgy3o8S9dhPDojKk3hHkluUkwwCwkboGxAGQKSg5HVYhxvXK0KBe8mndonXHruxr51h3gc/GNyLn0XHaz+JXtOAQssJHEOKRo5ec9uZgu0hXxqYIGWQgj0IlZsPBaN4ShTpNahflNIhYnoTRChk7NkiiTYnjIIaEF6Yx9ByBgYD8BMxIPIWYQ+LfkdqffI/x3RvUcMBIlxwpHJqt4MMzWaYyx8k3cUExI6VCGO7UNJKhUgRtxARjVf2xnc9SFsuBuDExfrpJQhPe4q0ZYlW7mnWD4TldNydd1ie5DBVz23Z6DlwhwrQFDGNj0GMMVuyatLHooWn2ZOENvYnVZ1vIrlCgpb/5JcbPrCdV0vsdTUW76HStfp+0dLnK7JnBHVExOdxvW5emIKImduH0Z3QG2VZe1we2iwwKDPtiYFIQy5xKsvogUDsDHhXRnbBb8xtDDidckKYB7zGt1bzVLoaP015G5l6w3thwdrMS+3J/Kh7cUGH90nZcXAP/ol1ebtINCIxLTws2c+R+IB6oDLopXEBzldtlKsXRFSyy80BhIleAMxsI8pdTXSEvj+Y6A81BAjkk2eBv/wb6jbEjiNmaH4Vy0CusgXGF1Idk8UEZ7cuVQs+InpcqgdYLFtxfT18IB/hbwP/Jq+WqJAAxoVR2dGzqc1hCA7dqnb0DlflFaHwODiDIEH+F8XdnnXHGn55p3Bv6YyMOwIi7zWJBhbqynFlW02BEE8mUMwmizRU7kEN0KANeOBBYUrzL0oSSe7Buwf0UX37ctcoWXoC1baBzkmtaQvgWUdltNi/1Ogf6Un030qfXKifpI4keDaOp0ktfGyFZGVro2aY+1BC0m0IO+UuGtpCwbs3xZVl+bbqx84NQAefStdoaNMRXdJgjDsRIHfeW1uo6WcLg9K8C+Ws9uxePYKD26QiRy9W6kYsxpY2DAAgNLvOsZnMIyphF/oZKxWx5NbuedGycU7qh5wYcYUzbZaSy5gNik5TlkycXv3JVZqXCqDaFsTmN2E80vR5O6nMgQaAWBdb1wQtOOoWa9dtvLH874ROBnqvNqo5NZJCRT+0pqvMYC8H67HF0tEUKGqroKptt0jB7ls0C9mIZyiEunfdQiVYEuAJ1WJfTVFxIjCExK97rrxau0NQ2mCSbxcLuAr9DLH0b+QBOsmcD4eEM2gs1UdurR8MhWBrG/UDIvsPwBEyDiG4NbBcSn4q8RcaOuWQA44Ai7pQy04TLRhvwkIX0WHQaN3ph89ERL+CPhcZxX8CHd1aLVZQuz7SNobIxfZw7zq76/a7/ECzbaYqAgclwcTDLzmspVFlE0hSoODbidK8UJuKzoJIPmka8AsiDMetlvMbsemBQLoErenQ0njf8FFIdYVLtf0w7cOgFtpGf0feng3DdKy8ku4LZXazAo2tEBe3IrZaG50FK1Wnrdgz26HI0dsUfRGrB0Z1Q+L6oPSvkFsXdShVRgA4lMb0SxWPKKwRzKXetthileVRED8qLci00cORttmmXVyEg4YuBBFSXPEU8wZjCaDDjFz5Q5jU+bVnzGU/NsboI3orX0Kf5a4s6rcOo5MLneT1KVRM9FLwS6qyrgTCmDWKLIfOO4O8+tFLQHz3Z7PjW8WWU9AAxOgTBJI8DmN2WMVEAgk6OPJjB8Lwlu9q7YyzfemTj8kBvmo4vMgQStLFMP+Yz+lvXWX02qEcJrLGDRN68PHRmNbwqSwezzyEJJCem+F/LSRgrOKwxHeWU1OEjQAWJ72sQnlK0iiFvPrtvA3SqpzbboNrzODLY/fTR88rhvPaj4m7W2DRka/Sn6Ojo1uCk1EJC2D3Fi4CZJuZ5usYTTtFuMLplKIlJluaRSbn6eVNO0/i8aUDoHOMheA86CT5+Lakk4+l6C8T1nFdSsgrkhKN3ZZ5dVNqqxIRBfaEPtczxtORXZvjFgHi+zjE9hCXWaFFShaoiGEG02y0gBpPEIXODugVvKo5DHM/h3pjCqixPE2+VrmjtqRgfvZa9JikE5hx+PA/M07XWTn+zwDQjdd/hc/nHKx2hgm0AGwAZ32ZBz9xAxPTRW0fBfaym9IrI6575BR78LL2tQnJhuckaGwZgp7tUBVEeX+H39v6T33XLF3zsAMCImJ6rNhCpZEugMfaGadzKrgnulsF5l41zlqYFL0WfAcNV6wzthKS5SIM0xlm4NoBdZulmjaqreyqwEwG05AmETNkICFsOMFLX/ku2llKQXy6kb1bZ27ESeRa4nAe4zTYF0wy3zhTqidecZRbAUf3T5vmaIOs+BUxC6m0LVETQ8eFDFKgK72ATSo8HmXb8udUxlEMG+RBBeoLcsq3bq5vz372zT5+u/undfjgce+df3n06++xd/ePi5ubj+wuL0nVrxPWr9m5sc/Q+nX25BBjvPl2d//3j5W/ljpaob5JcFSjSAWS6eBRWv6hPMpYbyJiEj9kK1WloI0uPL4N7yFJyuh2E4cVqE7ssncUCYEz5jXvHjiT5qbU+MOEOYpv7xlLmMyWbZ/RaXaS2Ji3GcFjSqg29NsjdYN0vWmLKChYGV3Veo4IzU1m+IVrUlRVehvU7UPc025ZRFwkbokZJpLcpAgzIdRuom6SPtukEdWHOgTwyRbWR1Jxi6aZJbl5DA3QnbWzjtD3uEvKcZm9Np8zH6umDfY1rYKp+s/pDLzcpVasaXU2lQlfzze6v9smmbYlukFe7tDJ8vvpyc37xGV+yWek9YyM6S69vrm6vzq8+eb+fff69/4ZoYM3+mUVtbgYMxolV60gH+PnvF+d/v776eHnbD567giiIi4ot7GvJw53MwkewFBVEM1KTwULFCqLobOv5CmhYW1zO2F3egTqCL1lSdmQBxBUYTPQr4DpW9Be9DkzqNtBB2cNm3o5qa3RZjAIPDyyF50sYnU0gBZgM54NvZUucXaFTx/oT7MLCl3b4+qVTT+g7Xd28R7HhVZRJY7rx4fpwjIEPWhbT+fLtc0nnvVLny03UJ+DF6kHhzSwFviireuDQDPIQtRkWd5lS4sO/2HpVLZ259e3D6cGtVA0s+urmAfPGxjrxVWGT+BZDMwzmvBXrBb2+oG5BtpSZSZgy7y6vXWabtOxG4ctFUuxzY6NuRIdUCk0zSAFEaGkSYycTJGHlrcmdkhmieMMV11EMMW6o1nG6xSFudnfFR4QdBdiTmD4KVh7hb/GBDIKGDaX1rriA5GGryyjiLs0LoTsPMXiu1c9CwQLNJfcFpHngqDEAjJUo81E2jXBk7HaIXkaTs/nr7t5Q4+Z1a6vHT3w5RDbPKFKP8E+55eO19/7iw6czfIpsNZJiuMa1Ykh7MGzTAruOIevigGQg+BV4NBCjw85bANbUeN2smFPuNDtNmm/unBaXPp/zviK9p8R+2yxLUG1S5xrs76hRk8vgVLuryuMcLnL0RIfPZ9ETpHs1z1YFueZwfPfI7aKWWxgyfNXRMVxVm4+eh6p6gZZKIhgxfy0L5xw0U5Gk/6nue4VYvjMFDvWabD0sKsmayHV+lxa1jLTBVbnUSFLZBv8588D5KlnQgPncg2gJeQZAMPPT7xcaeRjEvgqNvtNLI35hw9CWcsQGQPN40U+hViqwsG5filfv8iV59fTDgvcCUOBfOuPVolNaRWtKlrjN3+PG/ALt/FInHgnGEbrGoss6rZJKabn1QCVmA5D5nb2wFueaELFgIdA0F1TXx7rdoEYOLN9h28FOeKZTpAxImGk/BLtk948dUVY0v3XGuzb+jTNe208HKJvff3ZpP1xBfBDLlR/K0+/oQx/XTXUAY8hpX9VbP4t5q3xK31swkNszbYPn5YMDgXY6hcnWcoqO33CRcmdi3Lfn6PikWbDkyXrEPLN0ERO71qlvaVrWX9sgG9uwuMCVCLCiELWA17FMOcdUxFoAwBtUpnOv6vOuSfdgp2wOdotUmWno00pwVZWEsO1USnRjBxK0hsP8e5WTXno3UKC2LBADr1bVqLWG9Z7KeaSps2BbkMNDjb6yATZw/BgS7fPxJp5uHyyLUj+HSq358CcQqYJyClzo1G5U3tiCVbz8jpI4XP1jaRz/jqIMYVhJGaKLv9orfzVRqKfCxhE3hAgg13IKJjjBmG06NlWfRCa65IPlnf9J6r+qMb/kcy/xhwa3CosdoDQURYVRNqW6A4dPLpo7/LlTvllADja1XIih8BcNCpXSnAAR1KLyXbwTrk5J6IJ+zqCrAdm2Wtb/Ax/a+xPhWjcE6SHeoD8q7PG3ujRmOMFlt+oC38G7D7nv/TCJX3o1zc5vzmtPvGnuxlFyXxKlxpeysR+4vW2T3t0ktE+3vnnp/RRfVZxmNwaSx/QBSHw0ZTDUM6p/n+qeZcsNJgbX+C0DJ5MHWUS/wZp6XpgGnqdLd64MQwhReLVt7XNiso8lP6A+PZSiNFT1aJIN6+BeLiENOKDVlrMLWJmQASxNz5CuVAO9a6/u9Pv5jexRrQplzDXl6dDUGoEZ6HBcohTuzO2mO+yqEWi8Bwpmmr7Tos5SVz1FeZFzSQdirHgLaVKiHo1soLz1anGpwO+jjDNNmwSxWK3rv1HRRaaqaEssgEWG7Bn2rVifQbOpRfK36y+nRDY2a2UNA3wMk8ZDvTAGU+++1s+KVAvZ5zyH36UGYhXFcZRD3AD+qw655cVn336Ca553Tg+p2lWKyk0KdpP1Y37CtwIHDt6MrSZd5AP5sIYS7u1BsuMRITyP2r88D5XJ83TzF2vW3v8DyDWICg=='], 'pose_inference.py': ['3059a50208a07b76294f763f500c1043fce638b4902657380e9a6aa5188bfe23', 'eNqFVcGOpDYQvfMVFU4g9aDd60idQ0arSJF2s9JuchmNkBsK8IyxkW0m3fn6PNvQTfduZjggsMtV9V7VK+d5/ufkpdFCUcuTMqeRtSdvhXadsSPhRUKTUJZFeyJlRMvtjvhVqFl4bmkyjmmy3MrGG1vleZ7JcTLWU2OmU7b+YK8Zss6aMX1W3ZGWLXcaD0bJpkbUhrNkVAW/tfN9o1e7z8K9cPvt++8PX35TpnlZDdfYq91f2nk7N37GxrcBK1/h6utqlWVZyx3Vou8t94BQHxgguZ6seeYmcFEcgvv7HwKW9xnhAcTP5pXJD0zPRmp/J47S0Si8lUcaZ+XlpE6U3EargxTurrPM1AxCa1bExwkMI1YkLHiVHcWwlT7YWkldhUMEv9p4+mI0p+DhsUKC879RAf5krbFF/kdIg1ZIcAvyx3H27Mho5PKP9AOJTR4XsDSw5byMzmMBWtrflCTxkUwuBx3sHrVpOfZI/JB6cVH1VkxDFRZdQBY+KjPRfk95I5SqR9POinO0Vps2vbA9+2iwEJA/rbwo1sUmbkm/7OnjW3R8Ok6wBRLQRpplPxyQ4gZzSBR1ib0bU81v0QHcJuTjh5TM7NgG3Eo6v0mpiuvlNt+0EjMlxI6/8LLixHqSAboGHXNlAgv3+PFprf0lzFuQv69wYq1wEFLVPUO806QkqiDaZ1RGNycETuCTX7AUm5g6FkEy7kwFv0ozu6CLFmoC6jXDuB9b6qraUmPfS90vitoQVF5SP+sODq+Ox77oZp0kuGVnR8VtNomkD0+7DT3rWnlby7iBcMU59u7nGCuLEYiU6pAK0Lo6oNzi2MhkSZutwKAKPfxDkj+xRl/74mrdMqQ6ScXLsmVUYdXRMqzgeBJhRIWZKHUHxWrIEiJidf/OvCvp7td3TM5j7QFDHoUR26sgTPFUbNniX6JM5Hwwe+GTq+gBXEFnDe/IzN7BiLwc0VXnPLcjLqZcAZzUaJS3Gjr6DddMUcJ97Nh0+Vwum3XEJnrg7xJzaWKDu22U/8aZFoBULfMUPhJ3SxcE2ewJshTe2+J8Zkd5JBz+8l0cwWeFxyPQ581cTqU7O4gbcTiKEfQkZWL0pOkBF1XYaOtmkKpF2kW50QmiSAdJeaHXCbz7vxtpfdwCIfjeLVHfv+jKq8Y7Z18l8rP/AEtDyfc=']}
for name, (digest, packed) in sources.items():
    contents = zlib.decompress(base64.b64decode(packed))
    if hashlib.sha256(contents).hexdigest() != digest:
        raise ValueError("Embedded model transform or audit source differs")
    (code_dir / name).write_bytes(contents)
print("Verified paired runner and deployment transform.")

In [ ]:
out = Path("/kaggle/working/pinns_paired_pose_compute_source_label.zip")
subprocess.run([sys.executable, str(code_dir / "paired_pose_compute.py"),
                "--input-root", "/kaggle/input", "--transform", str(code_dir / "pose_inference.py"),
                "--device", "cuda:0", "--output", str(out)], check=True)